# Embedding a Knowledge Assistant in Microsoft Teams

This notebook covers two ways to put a Databricks Knowledge Assistant in front of Teams users. Both end at the same place: a working chatbot inside a Teams channel. They differ in how much work and polish is involved.

|  | Path 1: No-code | Path 2: Databricks App |
|---|---|---|
| Setup time | About 5 minutes | About 30 minutes |
| Custom code required | None | One small Streamlit file |
| End-user login | Each user signs into Databricks the first time | Optional, can be bypassed via service principal |
| Look and feel | Databricks workspace chrome appears inside the Teams tab | Clean branded chat panel, no Databricks UI |
| Branding (colors, logo) | None | Full control |
| Streaming responses | Yes (built in) | Yes (configured below) |
| Logging and audit | Built in via Knowledge Assistant traces | Plus whatever custom logging you add |
| When to choose | Quickest demo, internal users with Databricks access | Production rollout, polished UX, users without Databricks accounts |

Start with Path 1. Upgrade to Path 2 when one of the above limits becomes a problem.

## Prerequisites

- A deployed Knowledge Assistant agent in Databricks Agent Bricks. The agent must have a serving endpoint and a hosted URL.
- A Microsoft Teams workspace where you have permission to add tabs to channels.
- For Path 2 only: a Databricks workspace with Apps enabled.

## Placeholders used in this notebook

| Placeholder | Description |
|---|---|
| `<KA_ENDPOINT_NAME>` | The Knowledge Assistant serving endpoint name from Agent Bricks |
| `<KA_AGENT_URL>` | The hosted Knowledge Assistant chat URL inside Databricks |
| `<WORKSPACE_URL>` | Your Databricks workspace URL |
| `<APP_NAME>` | Name for the Databricks App you deploy in Path 2 |

# Path 1: No-code, use the Knowledge Assistant's built-in chat URL

Every Knowledge Assistant deployed in Agent Bricks has a hosted chat interface inside Databricks. The URL looks like:

```
https://<workspace>.cloud.databricks.com/ml/genai/agent-bricks/<agent-id>
```

This URL is just a webpage. You can add it as a Teams tab using the **Website** tab option in Teams. No code, no deployment, no configuration.

### Tradeoffs to know going in

- Each Teams user clicking the tab will be prompted to sign into Databricks the first time. They need a Databricks account in your workspace.
- The Databricks workspace top navigation and sidebar appear inside the Teams tab. The chat works fine, it just visually looks like "Databricks running inside Teams."
- You cannot change the colors, logo, or layout.

This is the right path for internal demos and for teams of users who already have Databricks workspace access. For external users or for a polished production rollout, see Path 2.

## Step 1.1: Find the Knowledge Assistant's chat URL

1. In Databricks, navigate to **Agent Bricks** in the left sidebar.
2. Click on your deployed Knowledge Assistant agent.
3. Copy the URL from the browser address bar while you are on the agent's page. It will look like:

   ```
   https://<workspace>.cloud.databricks.com/ml/genai/agent-bricks/<agent-id>
   ```

Save this URL. It is the only thing you need.

## Step 1.2: Add it as a Teams tab

1. Open Microsoft Teams.
2. Navigate to the channel where the Knowledge Assistant should live (e.g., the Data Governance channel).
3. At the top of the channel, click the **+ (Add a tab)** button.
4. Search for and select **Website**.
5. Set:
   - Tab name: `Knowledge Assistant`
   - URL: paste the URL from Step 1.1
6. Click **Save**.

The tab appears in the channel. Anyone in the channel who clicks the tab and signs into Databricks will see the chat interface and can interact with the Knowledge Assistant.

## Step 1.3: Test

Open the tab as a different Teams user, sign into Databricks when prompted, ask a test question. Confirm the answer arrives correctly with citations.

If this works for your use case, you are done. Skip the rest of this notebook unless you want the polish of Path 2.

# Path 2: Build a thin Databricks App as the chat surface

When Path 1's limitations matter (every user needs a Databricks login, Databricks chrome in the Teams tab, no branding control), Path 2 replaces the built-in chat URL with a small custom chat UI you deploy as a Databricks App.

The App is a single-page Streamlit chat interface that calls the Knowledge Assistant serving endpoint behind the scenes. It runs as a service principal, so end users do not need individual Databricks accounts. It streams responses for a fast, responsive feel. It hides all Databricks branding so the experience looks native to Teams.

What follows is the complete implementation: three files (`app.py`, `app.yaml`, `requirements.txt`), deploy steps, and Teams configuration.

## Step 2.1: Build the chat UI

Three files. Create a folder anywhere in your workspace, save these inside.

The Python cell below contains `app.py`. The Markdown cell after that contains the contents of `app.yaml` and `requirements.txt`.

The `app.py` code:

- Streams responses token-by-token so users see text appear as it is generated, with no waiting on the full response.
- Reuses a single HTTP connection across requests for low per-message latency.
- Hides the Streamlit menu, footer, and header so the UI looks clean.
- Handles errors gracefully with a clear message instead of a stack trace.
- Maintains conversation history within a session.

In [0]:
# app.py
# Minimal, fast, clean chat UI that calls a Knowledge Assistant serving endpoint.

import json
import os

import requests
import streamlit as st

WORKSPACE_URL    = os.environ['DATABRICKS_HOST'].rstrip('/')
KA_ENDPOINT_NAME = os.environ['KA_ENDPOINT_NAME']
TOKEN            = os.environ['DATABRICKS_TOKEN']

st.set_page_config(
    page_title='Knowledge Assistant',
    layout='centered',
    initial_sidebar_state='collapsed',
)

# Hide Streamlit chrome for a clean Teams-friendly look
st.markdown(
    '''
    <style>
      #MainMenu, footer, header { visibility: hidden; }
      .block-container { padding-top: 1rem; padding-bottom: 1rem; }
      [data-testid="stChatMessage"] { padding: 0.5rem 1rem; }
    </style>
    ''',
    unsafe_allow_html=True,
)

# Reused HTTP session for connection pooling (lower latency per turn)
if 'http' not in st.session_state:
    sess = requests.Session()
    sess.headers.update({
        'Authorization': f'Bearer {TOKEN}',
        'Content-Type': 'application/json',
    })
    st.session_state.http = sess

if 'messages' not in st.session_state:
    st.session_state.messages = []

st.title('Knowledge Assistant')

# Render conversation history
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])


def stream_response(messages):
    """Stream tokens from the Knowledge Assistant serving endpoint."""
    url = f'{WORKSPACE_URL}/serving-endpoints/{KA_ENDPOINT_NAME}/invocations'
    payload = {'messages': messages, 'stream': True}

    with st.session_state.http.post(url, json=payload, stream=True, timeout=120) as resp:
        if resp.status_code != 200:
            yield f'Error from Knowledge Assistant: HTTP {resp.status_code}'
            return

        for line in resp.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data: '):
                continue
            data_str = line[6:]
            if data_str == '[DONE]':
                break
            try:
                chunk = json.loads(data_str)
                delta = chunk.get('choices', [{}])[0].get('delta', {}).get('content', '')
                if delta:
                    yield delta
            except json.JSONDecodeError:
                continue


# Handle new user input
if prompt := st.chat_input('Ask a question'):
    st.session_state.messages.append({'role': 'user', 'content': prompt})
    with st.chat_message('user'):
        st.markdown(prompt)

    with st.chat_message('assistant'):
        try:
            answer = st.write_stream(stream_response(st.session_state.messages))
        except requests.exceptions.Timeout:
            answer = 'The request took too long. Please try a more focused question.'
            st.error(answer)
        except Exception as exc:
            answer = f'Something went wrong: {str(exc)[:200]}'
            st.error(answer)
        st.session_state.messages.append({'role': 'assistant', 'content': answer})

### Supporting files

Save the following two files alongside `app.py` in the same folder.

**`app.yaml`**

```yaml
command: ['streamlit', 'run', 'app.py', '--server.port', '8080', '--server.address', '0.0.0.0', '--server.headless', 'true']

env:
  - name: KA_ENDPOINT_NAME
    value: '<KA_ENDPOINT_NAME>'
```

**`requirements.txt`**

```text
streamlit==1.39.0
requests==2.32.3
```

Replace `<KA_ENDPOINT_NAME>` in `app.yaml` with your actual Knowledge Assistant serving endpoint name.

## Step 2.2: Deploy the App

1. In Databricks, click **Compute** > **Apps** in the left sidebar.
2. Click **Create app**.
3. Name it (e.g., `ka-chat`). Click **Create**.
4. Choose the workspace folder where your three files live (`app.py`, `app.yaml`, `requirements.txt`).
5. Click **Deploy**. Wait for the status to show "Running" (1-2 minutes the first time).

Once running, the app shows a URL at the top of its page:

```
https://<app-name>-<workspace-id>.<region>.databricksapps.com
```

Copy this URL. Open it in a new browser tab and verify the chat UI loads and answers a test question.

### Performance: keep the app warm

By default, Databricks Apps scale to zero when idle. The first request after idle hits a cold start of 10-30 seconds. For a production user-facing chatbot this is too slow.

Two options:

1. Configure the App's compute to stay always-on. In the app's Compute settings, set the minimum instances to 1. Slightly higher cost, no cold starts.
2. Schedule a cheap keep-alive ping every 5 minutes. Set up a Databricks Job that runs `curl https://<your-app-url>/healthz` on a 5-minute cron. Free, but adds a small orchestration dependency.

Pick option 1 for production. Use option 2 only if cost is a critical constraint.

## Step 2.3: Grant the App permission to call the Knowledge Assistant

The App runs as a service principal. That service principal needs `CAN_QUERY` permission on the Knowledge Assistant serving endpoint.

1. In Databricks, navigate to **Compute** > **Serving**.
2. Find the Knowledge Assistant endpoint (named `<KA_ENDPOINT_NAME>`).
3. Click the endpoint to open it.
4. Click the **Permissions** tab.
5. Click **Grant**. Select the App's service principal (its name will match `<APP_NAME>-app`).
6. Check `CAN_QUERY`. Click **Grant**.

If you skip this step, the App will return HTTP 403 when users ask questions.

## Step 2.4: Add the App as a Teams tab

Same Teams-side steps as Path 1, but with the App URL instead of the raw Knowledge Assistant URL.

1. Open Microsoft Teams.
2. Navigate to the channel where the Knowledge Assistant should live.
3. At the top of the channel, click **+ (Add a tab)**.
4. Search for and select **Website**.
5. Set:
   - Tab name: `Knowledge Assistant`
   - URL: the Databricks App URL from Step 2.2
6. Click **Save**.

The chat UI now appears as a Teams tab with clean branding, no Databricks chrome, and no per-user Databricks login prompt.

## Step 2.5: Test the end-user experience

1. Open the Teams channel from a colleague's account.
2. Click the Knowledge Assistant tab.
3. Confirm the chat loads quickly (sub-second, if the App is warm).
4. Ask three to five real questions. Confirm:
   - Responses stream in token-by-token, not as one big block.
   - Citations and source references appear correctly.
   - Conversation history persists across messages within a session.
   - Refreshing the tab clears the conversation (Streamlit session resets).

If responses feel slow, check:

- Is the app scaled to zero? See Step 2.2's keep-warm options.
- Is the Knowledge Assistant endpoint provisioning enough throughput? Check the endpoint's metrics panel.
- Is the user's network adding latency? Try from a fast network.

## When you want even more

If Path 2 still is not enough, two heavier alternatives exist. Neither is recommended unless a specific limitation forces it:

- **Native Microsoft Teams Bot (Bot Framework SDK).** Users can `@mention` the bot in any channel or DM, not just inside a specific channel tab. Requires Microsoft Bot Framework SDK, Azure Bot Service registration, and Azure hosting (Functions or App Service). Multi-week development effort. Use when stakeholders demand chat-anywhere instead of chat-in-a-tab.
- **Microsoft Copilot Studio custom skill.** Wraps the Knowledge Assistant endpoint as a Copilot skill, published to Teams via Copilot Studio. Adds Copilot Studio licensing. Use when the organization already invests in Copilot Studio and wants centralized agent governance there.

Both alternatives still point at the same Knowledge Assistant serving endpoint. Only the consumption layer changes.

## References

- [Knowledge Assistant](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant)
- [Databricks Apps overview](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/)
- [Streamlit in Databricks Apps](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/get-started/streamlit)
- [Model Serving endpoint permissions](https://docs.databricks.com/aws/en/machine-learning/model-serving/manage-serving-endpoints)
- [Adding Website tabs in Microsoft Teams](https://learn.microsoft.com/en-us/microsoftteams/manage-website-tab)